In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os
from PIL import Image
import copy

# --- CONFIGURATION ---
# Check for GPU (CUDA for Nvidia, ROCm for AMD if supported, or MPS for Mac)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters
IMG_SIZE = 100
BATCH_SIZE = 64  # Increased for stability
NUM_CLASSES = 33 # Matches your previous notebook
EPOCHS = 20
LEARNING_RATE = 0.0001 # Lower LR for smoother convergence

# Paths (Adjust these if your folder structure is different)
TRAIN_DIR = "Fruits/train/train" 
VAL_DIR = "Fruits/test"          # Using Test folder as Validation to prevent leakage

In [ ]:
# 1. Define Transforms
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        # --- AUGMENTATION START ---
        # These random changes force the model to learn features, not pixels
        transforms.RandomRotation(20),      
        transforms.RandomHorizontalFlip(),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        # --- AUGMENTATION END ---
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ]),
    'val': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ]),
}

# 2. Load Datasets
# We use ImageFolder to load images directly from directory structure
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=data_transforms['train'])
val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=data_transforms['val'])

# 3. Create DataLoaders
# num_workers=2 helps load data faster, set to 0 if on Windows and you get errors
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Classes: {train_dataset.classes}")

In [ ]:
class OptimizedFruitCNN(nn.Module):
    def __init__(self, num_classes):
        super(OptimizedFruitCNN, self).__init__()
        
        # Block 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32) # Batch Norm stabilizes learning
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)
        
        # Block 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        # Block 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        # Classifier Head
        # 100x100 -> pool -> 50x50 -> pool -> 25x25 -> pool -> 12x12
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(128 * 12 * 12, 512)
        self.dropout = nn.Dropout(0.5) # Disables 50% of neurons to prevent memorization
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        # Feature Extraction
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        
        # Classification
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = OptimizedFruitCNN(NUM_CLASSES).to(device)
print(model)

In [ ]:
# Setup Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Scheduler: Lower LR if validation loss doesn't improve for 3 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3, verbose=True)

# Tracking variables
train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_loss = float('inf')
early_stopping_patience = 5
no_improve_epochs = 0

print("Starting training...")

for epoch in range(EPOCHS):
    # --- TRAINING ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    avg_train_loss = running_loss / len(train_loader)
    avg_train_acc = correct / total
    
    # --- VALIDATION ---
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    avg_val_loss = val_loss / len(val_loader)
    avg_val_acc = correct / total
    
    # Store history
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    train_accs.append(avg_train_acc)
    val_accs.append(avg_val_acc)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} Acc: {avg_train_acc:.4f} | Val Loss: {avg_val_loss:.4f} Acc: {avg_val_acc:.4f}")
    
    # Step Scheduler
    scheduler.step(avg_val_loss)
    
    # Early Stopping Check
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_fruit_model.pth')
        no_improve_epochs = 0
    else:
        no_improve_epochs += 1
        if no_improve_epochs >= early_stopping_patience:
            print("Early stopping triggered!")
            break

# Load best model for use
model.load_state_dict(torch.load('best_fruit_model.pth'))
print("Training finished.")

In [ ]:
plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.title('Loss History')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Acc')
plt.plot(val_accs, label='Val Acc')
plt.title('Accuracy History')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
def predict_image(image_path, model, classes):
    # Load and transform image
    transform = data_transforms['val']
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probabilities, 1)
        
    predicted_class = classes[predicted_idx.item()]
    confidence_score = confidence.item() * 100
    
    # Show result
    plt.imshow(image)
    plt.title(f"Pred: {predicted_class} ({confidence_score:.1f}%)")
    plt.axis('off')
    plt.show()

# Example usage (Replace with a path to a real image)
# predict_image("downloads/my_apple.jpg", model, train_dataset.classes)